In [0]:
%python
import dlt
from pyspark.sql.functions import col, lit

# Define the schema for the CSV file
schema = """
ORDER_NUMBER INT,
RETAILER_NAME STRING,
RETAILER_SITE_CODE STRING,
RETAILER_CONTACT_CODE INT,
SALES_STAFF_CODE INT,
SALES_BRANCH_CODE INT,
ORDER_DATE DATE,
ORDER_CLOSE_DATE DATE,
ORDER_METHOD_CODE INT,
ORDER_DETAIL_CODE INT,
SHIP_DATE DATE,
PRODUCT_NUMBER STRING,
PROMOTION_CODE STRING,
QUANTITY INT,
UNIT_PRICE DOUBLE,
TOTAL_PRICE DOUBLE
"""

@dlt.table
def raw_transactions():
    return (
        spark.read.format("csv")
        .option("header", "true")
        .schema(schema)
        .load("path/to/your/transaction_record.csv")
    )

@dlt.table
def validated_products():
    product_numbers = dlt.read("raw_transactions").select("PRODUCT_NUMBER").distinct()
    existing_products = spark.table("tutorials.gosales.product").select("PRODUCT_NUMBER").distinct()
    
    new_products = product_numbers.subtract(existing_products)
    
    if new_products.count() > 0:
        new_product_records = new_products.withColumn("PRODUCT_TYPE_CODE", lit(None).cast("string")) \
                                          .withColumn("PRODUCT_BRAND_CODE", lit(None).cast("string")) \
                                          .withColumn("PRODUCT_COLOR_CODE", lit(None).cast("string")) \
                                          .withColumn("PRODUCT_SIZE_CODE", lit(None).cast("string")) \
                                          .withColumn("PRODUCT_LINE_CODE", lit(None).cast("string"))
        new_product_records.write.format("delta").mode("append").saveAsTable("tutorials.gosales.product")
    
    return spark.table("tutorials.gosales.product")

@dlt.table
def unique_order_header():
    raw_df = dlt.read("raw_transactions")
    existing_order_numbers = spark.table("tutorials.gosales.order_header").select("ORDER_NUMBER").distinct()
    
    new_order_header_df = raw_df.select(
        "ORDER_NUMBER",
        "RETAILER_NAME",
        "RETAILER_SITE_CODE",
        "RETAILER_CONTACT_CODE",
        "SALES_STAFF_CODE",
        "SALES_BRANCH_CODE",
        "ORDER_DATE",
        "ORDER_CLOSE_DATE",
        "ORDER_METHOD_CODE"
    ).distinct()
    
    unique_order_header_df = new_order_header_df.join(existing_order_numbers, "ORDER_NUMBER", "left_anti")
    
    return unique_order_header_df

@dlt.table
def unique_order_details():
    raw_df = dlt.read("raw_transactions")
    existing_order_details = spark.table("tutorials.gosales.order_details").select("ORDER_DETAIL_CODE").distinct()
    
    new_order_details_df = raw_df.select(
        "ORDER_DETAIL_CODE",
        "ORDER_NUMBER",
        "SHIP_DATE",
        "PRODUCT_NUMBER",
        "PROMOTION_CODE",
        "QUANTITY",
        "UNIT_PRICE",
        "TOTAL_PRICE"
    )
    
    unique_order_details_df = new_order_details_df.join(existing_order_details, "ORDER_DETAIL_CODE", "left_anti")
    
    return unique_order_details_df

In [0]:
%sql
-- Validate PRODUCT_NUMBER and create new records if necessary
MERGE INTO tutorials.gosales.product AS target
USING (
  SELECT DISTINCT PRODUCT_NUMBER
  FROM tutorials.gosales.staging_order_details
) AS source
ON target.PRODUCT_NUMBER = source.PRODUCT_NUMBER
WHEN NOT MATCHED THEN
  INSERT (PRODUCT_NUMBER, PRODUCT_TYPE_CODE, PRODUCT_BRAND_CODE, PRODUCT_COLOR_CODE, PRODUCT_SIZE_CODE, PRODUCT_LINE_CODE)
  VALUES (source.PRODUCT_NUMBER, NULL, NULL, NULL, NULL, NULL);

-- Merge new order details into the order_details table
MERGE INTO tutorials.gosales.order_details AS target
USING (
  SELECT
    ORDER_DETAIL_CODE,
    ORDER_NUMBER,
    SHIP_DATE,
    PRODUCT_NUMBER,
    PROMOTION_CODE,
    QUANTITY,
    UNIT_PRICE,
    TOTAL_PRICE
  FROM tutorials.gosales.staging_order_details
) AS source
ON target.ORDER_DETAIL_CODE = source.ORDER_DETAIL_CODE AND target.ORDER_NUMBER = source.ORDER_NUMBER
WHEN MATCHED THEN
  UPDATE SET
    target.SHIP_DATE = source.SHIP_DATE,
    target.PRODUCT_NUMBER = source.PRODUCT_NUMBER,
    target.PROMOTION_CODE = source.PROMOTION_CODE,
    target.QUANTITY = source.QUANTITY,
    target.UNIT_PRICE = source.UNIT_PRICE,
    target.TOTAL_PRICE = source.TOTAL_PRICE
WHEN NOT MATCHED THEN
  INSERT (
    ORDER_DETAIL_CODE,
    ORDER_NUMBER,
    SHIP_DATE,
    PRODUCT_NUMBER,
    PROMOTION_CODE,
    QUANTITY,
    UNIT_PRICE,
    TOTAL_PRICE
  )
  VALUES (
    source.ORDER_DETAIL_CODE,
    source.ORDER_NUMBER,
    source.SHIP_DATE,
    source.PRODUCT_NUMBER,
    source.PROMOTION_CODE,
    source.QUANTITY,
    source.UNIT_PRICE,
    source.TOTAL_PRICE
  );